# 4-Arm Union Benchmark — v1 Molmo2 LoRA, 4 sheets (GPU-only, Colab)

Clean, self-contained rebuild of the arms benchmark. Runs the real 4-arm union
(Arm 0 regex, Arm 1 whole-page Qwen, Arm 2 Molmo2+v1-adapter+crop, Arm 3 CV-hybrid Qwen)
on 4 real sheets — 2 GD + 2 PX, one big + one small each:

- `GD-B-540-DP-2920-005` (big)
- `GD-T-435-DT-2042-056` (small)
- `PX-2365-0140006-001` (big)
- `PX-2368-0180004-001` (small)

**Arm 2 uses the v1 Molmo2 pointing LoRA adapter** (`timthy45/molmo2-pnid-pointing-lora/v1`,
F1=0.732 on the 20-sheet Gupta gate — the best measured checkpoint; a later v2 attempt
regressed to F1=0.686 and is NOT used here), with the SAME class-agnostic prompt it was
actually fine-tuned on ("Point to every symbol in this P&ID tile."), batched across all 4
sheets' tiles so Molmo2 loads exactly once.

**Prior history this notebook consolidates** (previously scattered across
`ExtractionAgent_Local_GPUOnly.ipynb` + ad hoc scratchpad scripts, which got confusing):
- The original 4-arm run (2026-07-20) only completed on 2 PX sheets: revR 0.850 / 0.780,
  a +460%/+560% jump vs. Arm-1-alone. This notebook generalizes that to the 4 sheets above.
- Running Molmo2 and Qwen resident on GPU at the same time is FINE for one Molmo2 load
  (this succeeded before) — it only OOM'd on a SECOND Molmo2 reload (a masked-round-2
  experiment this notebook does not attempt). Arms are still run in the proven sequence:
  regex (no model) → Qwen for Arms 1 & 3 → Molmo2+v1 loaded once for Arm 2 pointing → freed
  → Qwen (already warm) for Arm 2's crop-reads.
- Every `(sheet, arm)` result is pushed to HF the instant it's computed and skipped on
  relaunch — repeated kernel/VM disconnects this session mean a dropped connection should
  never cost more than whichever single step was in flight.
- **Security note**: paste your OWN `HF_TOKEN` below. A previous notebook had a token
  hardcoded in plaintext — rotate that token if you haven't already.


In [1]:
!nvidia-smi

Wed Jul 22 11:34:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Config

In [2]:
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MOLMO_MODEL_ID = "allenai/Molmo2-O-7B"
QWEN_MAX_NEW_TOKENS = 4096

import os
# Token is read from the environment, never hardcoded. In Colab add it under
# Secrets (key: HF_TOKEN) and enable notebook access; locally just export it.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("HF_TOKEN is not set - add it to Colab Secrets or export it") from e
DATA_REPO = "timthy45/pnid-extraction-datasets"
EXTRACTION_AGENT_SRC_REPO = "timthy45/pnid-extraction-agent-src"
EXTRACTION_AGENT_SRC_FILE = "agent_src/latest.zip"

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "PASTE_YOUR_HF_TOKEN_HERE", "paste your HF token"

AG_DIR = "/content/sheets/AG_PNID"
RIVE_DIR = "/content/sheets/RIVE"

# full 13-sheet list (matches src/e2e_harness/score_revR_real_sheets.py verbatim) - only
# the 4 named in the intro above are actually used by the arms run at the bottom
SHEETS = [
    ("GD-B-540-DP-2920-005", f"{AG_DIR}/GD-B-540-DP-2920-005-Z.pdf"),
    ("GD-B-550-DP-3322-003", f"{AG_DIR}/GD-B-550-DP-3322-003-Z2.pdf"),
    ("GD-B-615-DP-1148-006", f"{AG_DIR}/GD-B-615-DP-1148-006-Z2.pdf"),
    ("GD-H-375-DP-2590-003", f"{AG_DIR}/GD-H-375-DP-2590-003-Zpdf.pdf"),
    ("GD-T-435-DR-2031-030", f"{AG_DIR}/GD-T-435-DR-2031-030-Z2.pdf"),
    ("GD-T-435-DT-2042-056", f"{AG_DIR}/GD-T-435-DT-2042-056-Z.pdf"),
    ("PX-2365-0140006-001", f"{RIVE_DIR}/PX-2365-0140006-001.PDF"),
    ("PX-2365-0140031-001", f"{RIVE_DIR}/PX-2365-0140031-001.PDF"),
    ("PX-2365-0150022-001", f"{RIVE_DIR}/PX-2365-0150022-001.pdf"),
    ("PX-2365-0150033-008", f"{RIVE_DIR}/PX-2365-0150033-008.pdf"),
    ("PX-2365-9850077-001", f"{RIVE_DIR}/PX-2365-9850077-001.pdf"),
    ("PX-2368-0180004-001", f"{RIVE_DIR}/PX-2368-0180004-001.pdf"),
    ("PX-2368-0180021-002", f"{RIVE_DIR}/PX-2368-0180021-002.pdf"),
]


## 2. Install (GPU-runtime prep)

In [3]:
!apt-get -qq update && apt-get -qq install -y libmagic1 > /dev/null
!pip install -q transformers==4.57.1 accelerate huggingface_hub
!pip install -q paddleocr paddlepaddle
!pip install -q pymupdf python-magic pydantic

# torchvision must match torch's own CUDA build, then pillow<12 pinned last (Pillow 12.0.0
# broke PIL._typing._Ink - known upstream regression).
import torch as _torch_pre
_ver, _, _local = _torch_pre.__version__.partition("+")
if _local.startswith("cu"):
    _idx = f"https://download.pytorch.org/whl/{_local}"
    !pip install -q torchvision "torch=={_ver}" --index-url {_idx}
else:
    !pip install -q torchvision "torch=={_ver}"

!pip install -q --force-reinstall --no-deps "pillow<12"

import importlib.metadata as _md
import PIL
from PIL import ImageDraw
import torchvision
from transformers import AutoModelForImageTextToText, AutoProcessor
import paddleocr

print("Pillow", PIL.__version__, "| torchvision", torchvision.__version__,
      "| transformers", _md.version("transformers"), "| paddleocr", _md.version("paddleocr"))
assert not PIL.__version__.startswith("12."), ("Pillow 12.x still present after pin - "
    "this kernel likely already had PIL imported from an earlier attempt in this "
    "session (a pip reinstall cannot hot-swap an already-imported module). "
    "Runtime -> Restart session, then rerun from the top.")
_disk_torch = _md.version("torch").split("+")[0]
if not _torch_pre.__version__.startswith(_disk_torch):
    raise RuntimeError(f"torch changed on disk ({_torch_pre.__version__} loaded vs "
                       f"{_disk_torch} installed) - Runtime -> Restart session, then rerun this cell once")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 151.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Private code: pnid-extraction-agent + e2e_bench + extraction_local

In [4]:
import zipfile, sys
from pathlib import Path
from huggingface_hub import hf_hub_download

AGENT_SRC_ROOT = Path("/content/agent_src")
AGENT_SRC_ROOT.mkdir(exist_ok=True)

zp = hf_hub_download(repo_id=EXTRACTION_AGENT_SRC_REPO, filename=EXTRACTION_AGENT_SRC_FILE,
                      repo_type="dataset", token=HF_TOKEN)
with zipfile.ZipFile(zp) as zf:
    zf.extractall(AGENT_SRC_ROOT)
print("extracted:", sorted(p.name for p in AGENT_SRC_ROOT.iterdir()))

AGENT_DIR = str(AGENT_SRC_ROOT / "agents" / "pnid-extraction-agent")
PID_ML_SRC = str(AGENT_SRC_ROOT / "pid_ml_src")
for p in (AGENT_DIR, PID_ML_SRC):
    if p not in sys.path:
        sys.path.insert(0, p)

!pip install -q pdfplumber

import importlib
REQUIRED_MODULES = [
    "pnid_pipeline.extract", "pnid_pipeline.vision", "pnid_pipeline.ocr_reasoning",
    "pnid_pipeline.grounded_read", "pnid_pipeline.triage", "pnid_pipeline.rasterize",
    "pnid_pipeline.run",
    "e2e_bench.backends.parse_json_common", "e2e_bench.backends.parse_molmo", "e2e_bench.types",
    "extraction_local.qwen_call_llm", "extraction_local.paddle_ocr",
    "extraction_local.molmo_candidates", "extraction_local.molmo_synthetic_tokens",
    "extraction_local.molmo_render", "extraction_local.run_extraction_local",
]
missing = []
for mod in REQUIRED_MODULES:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing.append(f"{mod}: {type(e).__name__}: {e}")
if missing:
    raise RuntimeError("Missing/broken imports:\n  " + "\n  ".join(missing))
print(f"all {len(REQUIRED_MODULES)} required modules import cleanly")

sys.path.insert(0, AGENT_DIR)
from scripts.eval.score import load_reviewed_truth, review_keep, review_recall
print("scripts.eval.score imported (real revR scorer)")

agent_src/latest.zip:   0%|          | 0.00/620k [00:00<?, ?B/s]

extracted: ['agents', 'pid_ml_src']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 151.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 157.7 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
all 16 required modules import cleanly
scripts.eval.score imported (real revR scorer)


## 4. Sheet PDFs — download

In [5]:
from huggingface_hub import hf_hub_download
import zipfile

for fname in ("AG_PNID.zip", "RIVE_LTTS_Sample.zip"):
    zp = hf_hub_download(repo_id=DATA_REPO, filename=f"sheets/{fname}",
                          repo_type="dataset", token=HF_TOKEN)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall("/content/sheets")
    print(f"extracted {fname}")

from pathlib import Path
missing_sheets = [(stem, p) for stem, p in SHEETS if not Path(p).exists()]
if missing_sheets:
    print(f"{len(missing_sheets)}/{len(SHEETS)} sheet PDFs not present:")
    for stem, p in missing_sheets:
        print(f"  MISSING  {stem}: {p}")
else:
    print(f"all {len(SHEETS)} sheet PDFs present.")

sheets/AG_PNID.zip:   0%|          | 0.00/63.0M [00:00<?, ?B/s]

extracted AG_PNID.zip


sheets/RIVE_LTTS_Sample.zip:   0%|          | 0.00/20.3M [00:00<?, ?B/s]

extracted RIVE_LTTS_Sample.zip
all 13 sheet PDFs present.


## 5. Load Qwen3-VL-8B (stays resident for the whole run)

In [6]:
from extraction_local.qwen_generate import load_qwen_model, build_qwen_generate_fn

qwen_model, qwen_processor = load_qwen_model(QWEN_MODEL_ID)
print("Qwen3-VL-8B (base) loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

qwen_generate_fn = build_qwen_generate_fn(qwen_model, qwen_processor,
                                           max_new_tokens_default=QWEN_MAX_NEW_TOKENS)
print("qwen_generate_fn ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen3-VL-8B (base) loaded. VRAM: 17.5 GB
qwen_generate_fn ready


In [7]:
from extraction_local.run_extraction_local import run_one_sheet
print("run_one_sheet imported")

run_one_sheet imported


## 6. Run the 4-arm union benchmark (Arm 1 → Arm 0 → Arm 3 → Arm 2 with v1 → final table)

Self-contained script, fetched fresh from HF each run (`force_download=True`) so it always
picks up the latest fix. Resumes automatically from any previously-completed `(sheet, arm)`
results pushed to `timthy45/pnid-extraction-agent-src/results/arms_final_v1_4sheets.json`.

In [13]:
from huggingface_hub import hf_hub_download

_p = hf_hub_download(repo_id=EXTRACTION_AGENT_SRC_REPO,
                     filename="colab_cells/arms_final_v1_4sheets.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

arms_final_v1_4sheets.py: 0.00B [00:00, ?B/s]

nest_asyncio applied (fixes asyncio.run() inside the Jupyter event loop)
torchao uninstalled (peft LoRA loader compatibility)


precomputed_ocr_words.json: 0.00B [00:00, ?B/s]

cached-OCR shim installed (stem-keyed)


arms_final_v1_4sheets.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

  v1 adapter loaded. VRAM: 43.4 GB
  1923 tiles queued across 4 sheet(s), batch_size=4
    batch 1/481  (1s elapsed)
    batch 21/481  (25s elapsed)
    batch 41/481  (67s elapsed)
    batch 61/481  (137s elapsed)
    batch 81/481  (179s elapsed)
    batch 101/481  (223s elapsed)
    batch 121/481  (250s elapsed)
    batch 141/481  (270s elapsed)
    batch 161/481  (293s elapsed)
    batch 181/481  (323s elapsed)
    batch 201/481  (345s elapsed)
    batch 221/481  (394s elapsed)
    batch 241/481  (450s elapsed)
    batch 261/481  (525s elapsed)
    batch 281/481  (567s elapsed)
    batch 301/481  (627s elapsed)
    batch 321/481  (688s elapsed)
    batch 341/481  (759s elapsed)
    batch 361/481  (819s elapsed)
    batch 381/481  (890s elapsed)
    batch 401/481  (944s elapsed)
    batch 421/481  (969s elapsed)
    batch 441/481  (989s elapsed)
    batch 461/481  (1066s elapsed)
    batch 481/481  (1127s elapsed)
  GD-B-540-DP-2920-005: 140 deduped points  (pushed to HF)
  GD-T-435-D

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Qwen reloaded. VRAM: 45.4 GB
  GD-B-540-DP-2920-005  arm2 revR=0.606 (66/109)  tags=91  pts=140  105s
  GD-T-435-DT-2042-056  arm2 revR=0.182 (2/11)  tags=11  pts=16  10s
  PX-2365-0140006-001  arm2 revR=0.524 (130/248)  tags=355  pts=771  582s
  PX-2368-0180004-001  arm2 revR=0.460 (23/50)  tags=63  pts=182  159s

══════════════════════════════════════════════════════════════════════════════
sheet                      size            arm0          arm1      arm2(v1)          arm3          UNION
GD-B-540-DP-2920-005       big    0.028 (3/109) 0.055 (6/109) 0.606 (66/109) 0.550 (60/109) 0.835 (91/109)
GD-T-435-DT-2042-056       small   0.000 (0/11)  0.818 (9/11)  0.182 (2/11)  0.636 (7/11)  0.909 (10/11)
PX-2365-0140006-001        big    0.149 (37/248) 0.040 (10/248) 0.524 (130/248) 0.669 (166/248) 0.843 (209/248)
PX-2368-0180004-001        small  0.200 (10/50)  0.120 (6/50) 0.460 (23/50) 0.540 (27/50)  0.720 (36/50)

raw per-arm texts saved to HF (timthy45/pnid-extraction-agent-src/r

In [11]:
!nvidia-smi
!kill -9 13697

Tue Jul 21 13:47:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   60C    P0            382W /  400W |   81091MiB /  81920MiB |     99%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [12]:
!nvidia-smi

Tue Jul 21 13:47:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   40C    P0             65W /  400W |   27068MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [14]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id=EXTRACTION_AGENT_SRC_REPO,
                     filename="colab_cells/arm2_6prompt_px.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

arm2_6prompt_px.py: 0.00B [00:00, ?B/s]

rendered PX-2368-0180004-001: 6120x3960
VRAM before Molmo2 load: 27.8 GB


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

v1 adapter loaded. VRAM: 43.5 GB
150 tiles x 6 classes = 900 generations, batch_size=4
  batch 1/225  (1s)
  batch 41/225  (105s)
  batch 81/225  (197s)
  batch 121/225  (248s)
  batch 161/225  (291s)
  batch 201/225  (353s)
pointing done: 327 raw points (pushed to HF)
Molmo2+v1 freed. VRAM: 27.8 GB
173 deduped points (vs 182 in the 1-prompt run, 378 in yesterday's zero-shot 6-prompt)
loading Qwen3-VL-8B for crop reads...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen loaded. VRAM: 45.4 GB
  reads 50/173  (44s)
  reads 100/173  (86s)
  reads 150/173  (130s)

arm2_6p (v1 + 6 prompts) revR=0.520 (26/50)  tags=59  pts=173


arms_final_v1_4sheets.json: 0.00B [00:00, ?B/s]

  arm0     revR=0.200 (10/50)
  arm1     revR=0.120 (6/50)
  arm2     revR=0.460 (23/50)
  arm2_6p  revR=0.520 (26/50)
  arm3     revR=0.540 (27/50)
  UNION (old: 1-prompt arm2)     revR=0.720 (36/50)
  UNION (new: 6-prompt arm2)     revR=0.760 (38/50)
  UNION (both arm2 variants)     revR=0.760 (38/50)

reference: yesterday's zero-shot 6-prompt union on this sheet = 0.780; GPT-5.5 = 0.98


In [7]:
from huggingface_hub import hf_hub_download
_p = hf_hub_download(repo_id=EXTRACTION_AGENT_SRC_REPO, filename="colab_cells/fill_point_grid.py",
                     repo_type="dataset", token=HF_TOKEN, force_download=True)
exec(open(_p).read())

fill_point_grid.py: 0.00B [00:00, ?B/s]

[13:14:15] need zeroshot 1p: []
[13:14:15] need zeroshot 6p: []
[13:14:15] need v1 6p: ['GD-B-540-DP-2920-005', 'PX-2365-0140006-001', 'GD-T-435-DT-2042-056']
[13:14:15] rendered GD-B-540-DP-2920-005: (11920, 8420)
[13:14:16] rendered GD-T-435-DT-2042-056: (5955, 4210)
[13:14:17] rendered PX-2365-0140006-001: (15120, 10800)
[13:14:17] loading Molmo2 + v1 LoRA...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[13:14:34] v1 loaded. VRAM: 31.2 GB
[13:15:21] 10638 generations queued (v1)
[13:15:23]   batch 1/1773  (1s)
[13:16:02]   batch 31/1773  (40s)
[13:16:42]   batch 61/1773  (81s)
[13:17:24]   batch 91/1773  (123s)
[13:18:06]   batch 121/1773  (164s)
[13:18:51]   batch 151/1773  (210s)
[13:19:50]   batch 181/1773  (268s)
[13:20:43]   batch 211/1773  (322s)
[13:21:37]   batch 241/1773  (376s)
[13:22:30]   batch 271/1773  (429s)
[13:23:21]   batch 301/1773  (480s)
[13:24:07]   batch 331/1773  (525s)
[13:24:56]   batch 361/1773  (575s)
[13:25:50]   batch 391/1773  (628s)
[13:26:39]   batch 421/1773  (677s)
[13:27:22]   batch 451/1773  (721s)
[13:28:03]   batch 481/1773  (761s)
[13:28:41]   batch 511/1773  (800s)
[13:29:20]   batch 541/1773  (839s)
[13:29:59]   batch 571/1773  (877s)
[13:30:49]   batch 601/1773  (928s)
[13:31:32]   batch 631/1773  (970s)
[13:32:12]   batch 661/1773  (1010s)
[13:33:13]   batch 691/1773  (1071s)
[13:34:11]   batch 721/1773  (1130s)
[13:35:15]   batch 751/1773  